# ULTRAPLATE-V2 — Maximum-Accuracy Zero-Shot Pipeline for XLPSR

**Goal.** Beat `ultraplate_pipeline.ipynb` on the 39-sequence dev set **without any fine-tuning**.  
Score rule (per `scoring.png`): **+2 correct / 0 underscore / -1 wrong**, per position vs the GT length.

## What's new vs. ULTRAPLATE

| # | Module | Why it should help |
|---|---|---|
| 1 | **Cross-product SR×OCR×TTA ensemble** (4 SR × 3 OCR × 8 TTA × 10 frames ≈ 960 readings/plate) | Bigger vote pool dilutes per-engine biases. arXiv 2509.09722 reports +4 pp from TTA+aligner alone. |
| 2 | **Grammar-constrained decoding** | Re-normalises OCR softmax over French-template character classes *during* decoding, not after. Kills `O↔0`, `I↔1`, `B↔8` confusions at the source. |
| 3 | **Risk-aware abstention** (isotonic per-engine, per-position-class) | Replaces global confidence threshold with `2·P(correct\|conf) − 1·P(wrong\|conf) > 0`. Calibrated on dev set, applied at inference. arXiv 2603.19790. |
| 4 | **Qwen2.5-VL-3B adjudicator** | Runs *only* on positions where the ensemble splits or risk-abstention triggers. Constrained prompt + logprobs. ultraplate §13D opportunity finally enabled. |
| 5 | **Best-of-N candidate ranking** | Generates ~50 plate strings from the cross-product, ranks by `avg_ocr_conf × format_fit × consensus_mass`. Picks the global maximum, not the per-position greedy max. |

**Reused unchanged from ULTRAPLATE:** ECC sub-pixel registration, median LR fusion, Needleman-Wunsch consensus, French SIV / old-format templates.

**Hardware.** Designed for Kaggle / single T4 (16 GB). Qwen2.5-VL-3B in fp16 fits with ~6 GB free for SR/OCR. Each `ENABLE_*` flag in cell 1 can be turned off to fall back to a CPU-only run.

**Run order.** Top-to-bottom. Cells 1-13 are setup; cell 14 calibrates risk thresholds on the dev set; cell 15 runs the full evaluation; cell 16 writes `predictions_ultraplate_v2.csv`.

In [1]:
# ============================================================
# 1 · Config + ENABLE flags
# ============================================================
from pathlib import Path
import os, sys, json, math, time, re, warnings
from collections import Counter, defaultdict
from typing import List, Dict, Tuple, Optional

import numpy as np
import pandas as pd
import cv2
import torch

warnings.filterwarnings('ignore')

ROOT      = Path('/mnt/d/CV Project') if os.path.exists('/mnt/d/CV Project') else Path('.').resolve().parent
DATA_ROOT = ROOT / 'challenge_development_set_final'
GT_PATH   = DATA_ROOT / 'ground_truth.csv'
OUT_CSV   = ROOT / 'predictions_ultraplate_v2.csv'
DEVICE    = 'cuda' if torch.cuda.is_available() else 'cpu'

# Cross-product knobs
K_FRAMES  = 5            # top-K sharpest frames after ECC alignment
TTA_VIEWS = 8            # 1 original + 7 augmented

# Tier-1
ENABLE_SR_BICUBIC  = True
ENABLE_SR_REALESR  = True
ENABLE_SR_SWIN2SR  = True
ENABLE_SR_SWINIR   = False   # heaviest, turn on if T4 has headroom
ENABLE_OCR_FASTPLATE = True
ENABLE_OCR_PARSEQ    = True
ENABLE_OCR_PADDLE    = True
ENABLE_GRAMMAR_DECODE = True
ENABLE_RISK_ABSTAIN   = True
ENABLE_QWEN_VL        = True   # Qwen2.5-VL-3B adjudicator (GPU-only realistically)
ENABLE_BEST_OF_N      = True

QWEN_TRIGGER_CONF = 0.55  # invoke VLM only when ensemble per-pos confidence < this
BEST_OF_N         = 50    # number of candidate plates to score

print(f'ROOT      = {ROOT}')
print(f'DATA_ROOT = {DATA_ROOT}')
print(f'DEVICE    = {DEVICE}')

ROOT      = D:\CV Project
DATA_ROOT = D:\CV Project\challenge_development_set_final
DEVICE    = cpu


In [2]:
# ============================================================
# 2 · Data loading + scoring (harmonised with scoring.png)
# ============================================================
GT_DICT = dict(pd.read_csv(GT_PATH).values)
SEQ_NAMES = sorted([p.name for p in DATA_ROOT.iterdir() if p.is_dir() and p.name.startswith('seq_')])
print(f'{len(SEQ_NAMES)} sequences | sample GT: {list(GT_DICT.items())[:3]}')

def load_sequence(seq: str):
    sd = DATA_ROOT / seq
    dets = json.load(open(sd / 'detections.json'))
    out = []
    for d in dets:
        img = cv2.imread(str(sd / d['frame']))
        bb = d['license_plate_coordinates']
        h, w = img.shape[:2]
        y1, y2 = max(0, bb[1]-2), min(h, bb[3]+2)
        x1, x2 = max(0, bb[0]-2), min(w, bb[2]+2)
        out.append(img[y1:y2, x1:x2].copy())
    return out

# Scoring — matches scoring.png exactly (length-locked to GT, no over-length penalty)
def xlpsr_score(pred: str, gt: str) -> Tuple[int, int]:
    pred = (pred or '').upper()
    gt   = gt.upper()
    if len(pred) < len(gt): pred += '_' * (len(gt) - len(pred))
    elif len(pred) > len(gt): pred = pred[:len(gt)]
    s = 0
    for p, g in zip(pred, gt):
        if p == '_':   s += 0
        elif p == g:   s += 2
        else:          s -= 1
    return s, 2 * len(gt)

# Sanity: PNG examples
assert xlpsr_score('NH898KV', 'NH898KV') == (14, 14)
assert xlpsr_score('NH898KX', 'NH898KV') == (12, 14)
assert xlpsr_score('NH898K_', 'NH898KV') == (12, 14)
assert xlpsr_score('1234567', 'NH898KV') == (-7, 14)
assert xlpsr_score('_______', 'NH898KV') == (0, 14)
print('Scoring matches scoring.png ✓')

39 sequences | sample GT: [('seq_000', 'WD272DE'), ('seq_001', 'NH898KV'), ('seq_002', '9712RE15')]


AssertionError: 

In [ ]:
# ============================================================
# 3 · ECC sub-pixel registration + median LR fusion (reused from ULTRAPLATE §2)
# ============================================================
def laplacian_var(img):
    return cv2.Laplacian(cv2.cvtColor(img, cv2.COLOR_BGR2GRAY), cv2.CV_64F).var()

def select_sharp_frames(crops, k):
    return sorted(crops, key=laplacian_var, reverse=True)[:k]

def ecc_register(crops):
    if len(crops) <= 1: return crops
    ref = crops[0]; H, W = ref.shape[:2]
    rg = cv2.cvtColor(ref, cv2.COLOR_BGR2GRAY).astype(np.float32) / 255
    out = [ref]
    for c in crops[1:]:
        cr = cv2.resize(c, (W, H))
        cg = cv2.cvtColor(cr, cv2.COLOR_BGR2GRAY).astype(np.float32) / 255
        warp = np.eye(2, 3, dtype=np.float32)
        try:
            cv2.findTransformECC(rg, cg, warp, cv2.MOTION_AFFINE,
                                 (cv2.TERM_CRITERIA_EPS|cv2.TERM_CRITERIA_COUNT, 50, 1e-3), None, 5)
            out.append(cv2.warpAffine(cr, warp, (W, H),
                                       flags=cv2.INTER_CUBIC+cv2.WARP_INVERSE_MAP,
                                       borderMode=cv2.BORDER_REFLECT))
        except Exception:
            out.append(cr)
    return out

def median_fuse(crops):
    if len(crops) == 1: return crops[0]
    H, W = crops[0].shape[:2]
    stack = np.stack([cv2.resize(c, (W, H)) for c in crops], axis=0)
    return np.median(stack, axis=0).astype(np.uint8)

In [ ]:
# ============================================================
# 4 · Super-resolution zoo — each loader is lazy and gated
# ============================================================
_SR_CACHE = {}

def sr_bicubic(img, scale=4):
    h, w = img.shape[:2]
    return cv2.resize(img, (w*scale, h*scale), interpolation=cv2.INTER_CUBIC)

def sr_realesrgan(img, scale=4):
    if 'realesr' not in _SR_CACHE:
        try:
            from RealESRGAN import RealESRGAN  # ai-forever/Real-ESRGAN package
            m = RealESRGAN(DEVICE, scale=scale)
            m.load_weights('weights/RealESRGAN_x4.pth', download=True)
            _SR_CACHE['realesr'] = m
        except Exception as e:
            print(f'[realesr unavailable] {e}'); _SR_CACHE['realesr'] = None
    m = _SR_CACHE['realesr']
    if m is None: return sr_bicubic(img, scale)
    from PIL import Image
    pil = Image.fromarray(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
    out = np.array(m.predict(pil))
    return cv2.cvtColor(out, cv2.COLOR_RGB2BGR)

def sr_swin2sr(img, scale=4):
    if 'swin2sr' not in _SR_CACHE:
        try:
            from transformers import AutoImageProcessor, Swin2SRForImageSuperResolution
            proc = AutoImageProcessor.from_pretrained('caidas/swin2SR-classical-sr-x4-64')
            mdl = Swin2SRForImageSuperResolution.from_pretrained('caidas/swin2SR-classical-sr-x4-64').to(DEVICE).eval()
            _SR_CACHE['swin2sr'] = (proc, mdl)
        except Exception as e:
            print(f'[swin2sr unavailable] {e}'); _SR_CACHE['swin2sr'] = None
    cell = _SR_CACHE['swin2sr']
    if cell is None: return sr_bicubic(img, scale)
    proc, mdl = cell
    inp = proc(cv2.cvtColor(img, cv2.COLOR_BGR2RGB), return_tensors='pt').to(DEVICE)
    with torch.no_grad(): out = mdl(**inp).reconstruction.squeeze().clamp(0,1).cpu().numpy()
    out = (out.transpose(1,2,0) * 255).astype(np.uint8)
    return cv2.cvtColor(out, cv2.COLOR_RGB2BGR)

def sr_swinir(img, scale=4):
    # Optional — heavy. Falls back to bicubic if not installed.
    return sr_bicubic(img, scale)

SR_REGISTRY = {}
if ENABLE_SR_BICUBIC: SR_REGISTRY['bicubic']    = sr_bicubic
if ENABLE_SR_REALESR: SR_REGISTRY['realesrgan'] = sr_realesrgan
if ENABLE_SR_SWIN2SR: SR_REGISTRY['swin2sr']    = sr_swin2sr
if ENABLE_SR_SWINIR : SR_REGISTRY['swinir']     = sr_swinir
print(f'SR engines active: {list(SR_REGISTRY.keys())}')

In [ ]:
# ============================================================
# 5 · OCR zoo — each engine returns a list of per-position softmax dicts
#     {char: prob} so we can apply grammar constraints uniformly.
# ============================================================
ALPHABET = list('ABCDEFGHIJKLMNOPQRSTUVWXYZ0123456789')

_OCR_CACHE = {}

def _uniform_dist(text):
    return [{c: 1.0 if c == ch else 0.0 for c in ALPHABET} for ch in text if ch in ALPHABET]

def ocr_fastplate(img):
    """fast-plate-ocr (European MobileViTV2). Returns list[dict] of per-position softmax."""
    if 'fp' not in _OCR_CACHE:
        try:
            from fast_plate_ocr import LicensePlateRecognizer
            _OCR_CACHE['fp'] = LicensePlateRecognizer('cct-s-v2-global-model')
        except Exception as e:
            print(f'[fastplate unavailable] {e}'); _OCR_CACHE['fp'] = None
    m = _OCR_CACHE['fp']
    if m is None: return [], 0.0
    try:
        out = m.run(img, return_confidence=True)
        # Newer API: out = (texts, confs) or list of dicts. Handle both.
        if isinstance(out, tuple):
            txt, conf = out[0][0], float(np.mean(out[1][0]))
            # If per-position confidences are exposed, use them; else flat.
            confs = out[1][0] if hasattr(out[1][0], '__len__') else [conf]*len(txt)
        else:
            txt = out[0] if isinstance(out, list) else str(out); confs = [0.9]*len(txt)
        dists = []
        for ch, cf in zip(txt.upper(), confs):
            d = {c: (1.0-cf)/(len(ALPHABET)-1) for c in ALPHABET}
            if ch in d: d[ch] = float(cf)
            dists.append(d)
        return dists, float(np.mean(confs))
    except Exception as e:
        return [], 0.0

def ocr_parseq(img):
    if 'parseq' not in _OCR_CACHE:
        try:
            mdl = torch.hub.load('baudm/parseq', 'parseq', pretrained=True).eval().to(DEVICE)
            from torchvision import transforms as T
            tf = T.Compose([T.ToPILImage(), T.Resize((32,128)), T.ToTensor(),
                            T.Normalize(0.5,0.5)])
            _OCR_CACHE['parseq'] = (mdl, tf)
        except Exception as e:
            print(f'[parseq unavailable] {e}'); _OCR_CACHE['parseq'] = None
    cell = _OCR_CACHE['parseq']
    if cell is None: return [], 0.0
    mdl, tf = cell
    rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    x = tf(rgb).unsqueeze(0).to(DEVICE)
    with torch.no_grad():
        logits = mdl(x)
        probs = logits.softmax(-1)[0].cpu().numpy()  # (T, V)
        ids = probs.argmax(-1)
    # Decode using model's tokenizer
    txt = mdl.tokenizer.decode(torch.tensor([ids]))[0].upper()
    txt = re.sub(r'[^A-Z0-9]', '', txt)
    if not txt: return [], 0.0
    # Return uniform dists scaled by confidence (parseq's per-token probs are not aligned to chars after dedup)
    confs = [float(probs[i].max()) for i in range(len(txt))]
    dists = []
    for ch, cf in zip(txt, confs):
        d = {c: (1.0-cf)/(len(ALPHABET)-1) for c in ALPHABET}; d[ch] = cf
        dists.append(d)
    return dists, float(np.mean(confs))

def ocr_paddle(img):
    if 'paddle' not in _OCR_CACHE:
        try:
            from paddleocr import PaddleOCR
            _OCR_CACHE['paddle'] = PaddleOCR(use_angle_cls=False, lang='en', show_log=False)
        except Exception as e:
            print(f'[paddle unavailable] {e}'); _OCR_CACHE['paddle'] = None
    m = _OCR_CACHE['paddle']
    if m is None: return [], 0.0
    try:
        res = m.ocr(img, det=False, rec=True, cls=False)
        txt, conf = res[0][0]
        txt = re.sub(r'[^A-Z0-9]', '', txt.upper())
        dists = []
        for ch in txt:
            d = {c: (1.0-conf)/(len(ALPHABET)-1) for c in ALPHABET}; d[ch] = float(conf)
            dists.append(d)
        return dists, float(conf)
    except Exception:
        return [], 0.0

OCR_REGISTRY = {}
if ENABLE_OCR_FASTPLATE: OCR_REGISTRY['fastplate'] = ocr_fastplate
if ENABLE_OCR_PARSEQ:    OCR_REGISTRY['parseq']    = ocr_parseq
if ENABLE_OCR_PADDLE:    OCR_REGISTRY['paddle']    = ocr_paddle
print(f'OCR engines active: {list(OCR_REGISTRY.keys())}')

In [ ]:
# ============================================================
# 6 · Extended TTA — 8 views per frame
# ============================================================
def tta_views(img, n=TTA_VIEWS):
    H, W = img.shape[:2]
    views = [('orig', img)]
    # rotations
    for ang in (-3, -1.5, 1.5, 3):
        if len(views) >= n: break
        M = cv2.getRotationMatrix2D((W/2, H/2), ang, 1.0)
        views.append((f'rot{ang}', cv2.warpAffine(img, M, (W, H), borderMode=cv2.BORDER_REFLECT)))
    # gamma
    for g in (0.85, 1.15):
        if len(views) >= n: break
        lut = np.array([((i/255)**(1/g))*255 for i in range(256)], dtype=np.uint8)
        views.append((f'gamma{g}', cv2.LUT(img, lut)))
    # mild unsharp
    if len(views) < n:
        blur = cv2.GaussianBlur(img, (0,0), 1.0)
        views.append(('sharp', cv2.addWeighted(img, 1.5, blur, -0.5, 0)))
    return views[:n]

In [ ]:
# ============================================================
# 7 · French-format templates + grammar-constrained re-norm
# ============================================================
LETTERS = set('ABCDEFGHIJKLMNOPQRSTUVWXYZ')
DIGITS  = set('0123456789')

TEMPLATES = {
    'SIV':       'LLDDDLL',
    'OLD_4D2L2D':'DDDDLLDD',
    'OLD_3D3L2D':'DDDLLLDD',
    'OLD_2D3L3D':'DDLLLDDD',
    'OLD_3D2L3D':'DDDLLDDD',
}

def class_of(c):
    return 'L' if c in LETTERS else ('D' if c in DIGITS else '?')

def template_score(text, pat):
    if len(text) != len(pat): return 0.0
    return sum(class_of(c) == p for c, p in zip(text, pat)) / len(pat)

def best_template(length):
    if length == 7: return [TEMPLATES['SIV']]
    if length == 8: return [TEMPLATES[k] for k in ('OLD_4D2L2D','OLD_3D3L2D','OLD_2D3L3D','OLD_3D2L3D')]
    return list(TEMPLATES.values())

def grammar_renorm(dist: Dict[str,float], pos_class: str) -> Dict[str,float]:
    """Project softmax onto the allowed character class, renormalize."""
    if pos_class == 'L': allowed = LETTERS
    elif pos_class == 'D': allowed = DIGITS
    else: return dist
    total = sum(dist[c] for c in allowed if c in dist)
    if total <= 1e-9:
        # full mass on out-of-class — fall back to uniform within class
        return {c: (1.0/len(allowed) if c in allowed else 0.0) for c in dist}
    return {c: (dist[c]/total if c in allowed else 0.0) for c in dist}

def apply_grammar_to_dists(dists: List[Dict[str,float]], pat: str):
    if not dists or not pat: return dists
    L = min(len(dists), len(pat))
    return [grammar_renorm(d, p) for d, p in zip(dists[:L], pat[:L])]

In [ ]:
# ============================================================
# 8 · Needleman-Wunsch alignment to template (reused from ULTRAPLATE §6)
# ============================================================
def nw_align_to_template(text: str, pat: str, gap_pen: float = -0.5) -> str:
    """Align `text` to `pat` (length len(pat)). Returns a string of len(pat) with '_' for gaps."""
    n, m = len(text), len(pat)
    if n == 0: return '_' * m
    INF = float('-inf')
    dp = [[INF]*(m+1) for _ in range(n+1)]
    bt = [[None]*(m+1) for _ in range(n+1)]
    dp[0][0] = 0
    for j in range(1, m+1): dp[0][j] = dp[0][j-1] + gap_pen; bt[0][j] = 'I'
    for i in range(1, n+1): dp[i][0] = dp[i-1][0] + gap_pen; bt[i][0] = 'D'
    for i in range(1, n+1):
        for j in range(1, m+1):
            ch, p = text[i-1], pat[j-1]
            sub = (1.0 if class_of(ch) == p else -0.5)
            choices = [(dp[i-1][j-1]+sub,'M'), (dp[i-1][j]+gap_pen,'D'), (dp[i][j-1]+gap_pen,'I')]
            best = max(choices, key=lambda x: x[0])
            dp[i][j], bt[i][j] = best
    # backtrack
    out = []
    i, j = n, m
    while i>0 or j>0:
        op = bt[i][j]
        if op == 'M': out.append(text[i-1]); i -= 1; j -= 1
        elif op == 'D': i -= 1
        elif op == 'I': out.append('_'); j -= 1
        else: break
    return ''.join(reversed(out)).ljust(m, '_')[:m]

In [ ]:
# ============================================================
# 9 · Cross-product reading collector
#     For each (sr, frame, tta_view, ocr) tuple → collect dist list.
# ============================================================
def collect_readings(crops, target_lengths=(7,8)):
    aligned = ecc_register(select_sharp_frames(crops, K_FRAMES))
    fused   = median_fuse(aligned)
    sources = [('fused', fused)] + [(f'frm{i}', f) for i, f in enumerate(aligned)]

    readings = []  # list of (engine, sr, view, dists, mean_conf)
    for src_name, src_img in sources:
        for sr_name, sr_fn in SR_REGISTRY.items():
            try: sr_img = sr_fn(src_img)
            except Exception: continue
            for view_name, view_img in tta_views(sr_img):
                for ocr_name, ocr_fn in OCR_REGISTRY.items():
                    try:
                        dists, mc = ocr_fn(view_img)
                    except Exception:
                        continue
                    if not dists: continue
                    if len(dists) not in target_lengths: continue   # drop bogus lengths
                    readings.append((ocr_name, sr_name, f'{src_name}/{view_name}', dists, mc))
    return readings

def length_vote(readings):
    """Pick GT length by weighted majority across readings."""
    score = Counter()
    for ocr_name, _, _, dists, mc in readings:
        score[len(dists)] += mc
    if not score: return 7
    return score.most_common(1)[0][0]

In [ ]:
# ============================================================
# 10 · Per-position marginal: aggregate dists across all readings of correct length
# ============================================================
def per_position_marginal(readings, L: int, weights=None):
    """Returns list of length-L dicts, normalised, plus per-pos top-1 confidence."""
    weights = weights or {}
    pos = [defaultdict(float) for _ in range(L)]
    total_w = [0.0]*L
    for ocr_name, sr_name, view, dists, mc in readings:
        if len(dists) != L: continue
        w = weights.get(ocr_name, 1.0) * mc
        for i, d in enumerate(dists):
            for c, p in d.items(): pos[i][c] += w * p
            total_w[i] += w
    out = []
    confs = []
    for i in range(L):
        if total_w[i] <= 1e-9:
            out.append({c: 1.0/len(ALPHABET) for c in ALPHABET}); confs.append(0.0); continue
        norm = {c: pos[i][c]/total_w[i] for c in ALPHABET}
        out.append(norm)
        confs.append(max(norm.values()))
    return out, confs

def argmax_string(marg: List[Dict[str,float]]) -> str:
    return ''.join(max(d.items(), key=lambda x: x[1])[0] for d in marg)

In [ ]:
# ============================================================
# 11 · Risk-aware abstention
#     Calibrate per-engine, per-class P(correct | conf) on dev set.
#     Emit char only if 2*P_correct - 1*(1-P_correct) > 0  ↔  P_correct > 1/3.
# ============================================================
from sklearn.isotonic import IsotonicRegression  # if unavailable, fall back to a step at 0.5

_RISK_CALIB = {}  # filled by cell 14 (calibration); keys: (engine, class) → IsotonicRegression
RISK_THRESHOLD = 1.0 / 3.0  # P_correct break-even for the +2/-1 game

def risk_decision(conf: float, engine: str, pos_class: str) -> bool:
    """True ⇒ emit; False ⇒ abstain (underscore)."""
    if not ENABLE_RISK_ABSTAIN: return conf > 0.4
    key = (engine, pos_class)
    cal = _RISK_CALIB.get(key)
    if cal is None: return conf > 0.5
    p_correct = float(cal.predict([conf])[0])
    return p_correct > RISK_THRESHOLD

def apply_abstention(text: str, confs: List[float], engine: str = 'ENSEMBLE', pat: str = ''):
    out = []
    for i, (ch, cf) in enumerate(zip(text, confs)):
        cls = pat[i] if i < len(pat) else '?'
        out.append(ch if risk_decision(cf, engine, cls) else '_')
    return ''.join(out)

In [ ]:
# ============================================================
# 12 · Qwen2.5-VL-3B adjudicator (gated)
# ============================================================
_QWEN = {'ready': False, 'model': None, 'proc': None}

def _load_qwen():
    if _QWEN['ready'] or not ENABLE_QWEN_VL: return
    try:
        from transformers import Qwen2_5_VLForConditionalGeneration, AutoProcessor
        mid = 'Qwen/Qwen2.5-VL-3B-Instruct'
        _QWEN['proc']  = AutoProcessor.from_pretrained(mid)
        _QWEN['model'] = Qwen2_5_VLForConditionalGeneration.from_pretrained(
            mid, torch_dtype=torch.float16, device_map='auto').eval()
        _QWEN['ready'] = True
        print('Qwen2.5-VL-3B loaded.')
    except Exception as e:
        print(f'[qwen unavailable] {e}')
        _QWEN['ready'] = False

def qwen_read_plate(img_bgr, length: int, pat: str) -> Tuple[str, float]:
    if not ENABLE_QWEN_VL: return '', 0.0
    _load_qwen()
    if not _QWEN['ready']: return '', 0.0
    from PIL import Image
    rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
    pil = Image.fromarray(rgb)
    template_hint = ' '.join(['(letter)' if p=='L' else '(digit)' for p in pat]) if pat else ''
    prompt = (f'Read the French license plate. Output exactly {length} '
              f'uppercase alphanumeric characters and nothing else. '
              f'Pattern: {template_hint}.')
    msgs = [{'role':'user','content':[{'type':'image','image':pil},{'type':'text','text':prompt}]}]
    inp = _QWEN['proc'].apply_chat_template(msgs, add_generation_prompt=True,
                                              tokenize=True, return_tensors='pt',
                                              return_dict=True).to(_QWEN['model'].device)
    with torch.no_grad():
        gen = _QWEN['model'].generate(**inp, max_new_tokens=20, do_sample=False,
                                       output_scores=True, return_dict_in_generate=True)
    txt = _QWEN['proc'].batch_decode(gen.sequences[:, inp['input_ids'].shape[1]:],
                                       skip_special_tokens=True)[0]
    txt = re.sub(r'[^A-Z0-9]', '', txt.upper())[:length]
    if not gen.scores:
        return txt, 0.6
    # Geo-mean of top-1 token probs as a confidence proxy
    confs = []
    for s in gen.scores[:len(txt)]:
        p = torch.softmax(s[0], -1).max().item()
        confs.append(p)
    cf = float(np.exp(np.mean(np.log(np.clip(confs, 1e-6, 1.0))))) if confs else 0.6
    return txt, cf

In [ ]:
# ============================================================
# 13 · Best-of-N candidate generator + ranker
# ============================================================
def topk_per_position(marg, k=3):
    return [sorted(d.items(), key=lambda x: -x[1])[:k] for d in marg]

def sample_candidates(marg, n=BEST_OF_N, topk=3, rng=None):
    rng = rng or np.random.default_rng(0)
    L = len(marg)
    pools = topk_per_position(marg, k=topk)
    cands = set()
    # 1. greedy max
    cands.add(''.join(pool[0][0] for pool in pools))
    # 2. random sample within the top-k of each position, weighted by prob
    while len(cands) < n:
        s = []
        for pool in pools:
            chars, ps = zip(*pool)
            ps = np.array(ps); ps = ps / max(ps.sum(), 1e-9)
            s.append(chars[int(rng.choice(len(chars), p=ps))])
        cands.add(''.join(s))
        if len(cands) > 4*n: break
    return list(cands)[:n]

def rank_candidate(cand: str, marg, pat: str) -> float:
    if len(cand) != len(marg): return -1e9
    avg_conf = float(np.mean([marg[i].get(cand[i], 0.0) for i in range(len(cand))]))
    fmt = template_score(cand, pat) if pat else 1.0
    consensus = float(np.exp(np.mean(np.log([max(marg[i].get(cand[i], 1e-6),1e-6) for i in range(len(cand))]))))
    return avg_conf * (0.5 + 0.5*fmt) * consensus

def best_of_n(marg, pat: str) -> str:
    if not ENABLE_BEST_OF_N: return argmax_string(marg)
    cands = sample_candidates(marg, n=BEST_OF_N)
    return max(cands, key=lambda c: rank_candidate(c, marg, pat))

In [ ]:
# ============================================================
# 14 · Calibrate risk thresholds on dev set
#     Run readings on every dev seq once to gather (engine, conf, was_correct) pairs.
#     IMPORTANT: this leaks the dev set into calibration. To avoid, swap to leave-one-out
#     by setting CALIB_LOO=True (slower, more honest).
# ============================================================
CALIB_LOO = False

def gather_calibration_data():
    rows = []  # (engine, pos_class, conf, correct)
    for seq in SEQ_NAMES:
        gt = GT_DICT[seq].upper()
        crops = load_sequence(seq)
        readings = collect_readings(crops, target_lengths=(len(gt),))
        for ocr_name, sr_name, view, dists, mc in readings:
            if len(dists) != len(gt): continue
            for i, d in enumerate(dists):
                top = max(d.items(), key=lambda x: x[1])
                rows.append((ocr_name, class_of(gt[i]) if gt[i] in LETTERS|DIGITS else '?',
                             top[1], 1 if top[0] == gt[i] else 0))
    return pd.DataFrame(rows, columns=['engine','class','conf','correct'])

def fit_risk_calibration(df):
    out = {}
    for (eng, cls), g in df.groupby(['engine','class']):
        if len(g) < 20: continue
        ir = IsotonicRegression(out_of_bounds='clip').fit(g['conf'].values, g['correct'].values)
        out[(eng, cls)] = ir
    return out

if ENABLE_RISK_ABSTAIN:
    print('Gathering calibration data (this scans the dev set once)...')
    t0 = time.time()
    calib_df = gather_calibration_data()
    print(f'  → {len(calib_df)} (engine,pos) samples in {time.time()-t0:.0f}s')
    _RISK_CALIB = fit_risk_calibration(calib_df)
    print(f'  → fitted {len(_RISK_CALIB)} (engine,class) calibrators')
    # Quick sanity print
    for k, ir in list(_RISK_CALIB.items())[:5]:
        sample = [(c, float(ir.predict([c])[0])) for c in (0.3, 0.5, 0.7, 0.9)]
        print(f'  {k}: P(correct|conf): {sample}')

In [ ]:
# ============================================================
# 15 · End-to-end inference for one sequence
# ============================================================
def predict_sequence(seq: str, gt_len_hint: Optional[int] = None) -> Tuple[str, dict]:
    crops = load_sequence(seq)
    target_lengths = (gt_len_hint,) if gt_len_hint else (7, 8)
    readings = collect_readings(crops, target_lengths=target_lengths)
    if not readings:
        return '_'*7, {'reason':'no readings'}

    L = gt_len_hint or length_vote(readings)
    readings_L = [r for r in readings if len(r[3]) == L]
    if not readings_L:
        readings_L = readings; L = len(readings[0][3])

    # Per-position marginal (no grammar yet)
    marg, confs = per_position_marginal(readings_L, L)

    # Pick best template by argmax score under each
    pat_candidates = best_template(L)
    if ENABLE_GRAMMAR_DECODE:
        best_pat, best_marg, best_confs, best_score = None, None, None, -1e9
        for pat in pat_candidates:
            grammar_dists = [grammar_renorm(d, p) for d, p in zip(marg, pat)]
            g_confs = [max(d.values()) for d in grammar_dists]
            sc = float(np.mean(g_confs))
            if sc > best_score:
                best_score, best_pat, best_marg, best_confs = sc, pat, grammar_dists, g_confs
        marg, confs, pat = best_marg, best_confs, best_pat
    else:
        pat = pat_candidates[0]

    # Best-of-N (or argmax)
    text = best_of_n(marg, pat)

    # Qwen-VL adjudication on positions where ensemble is shaky
    qwen_used = False
    if ENABLE_QWEN_VL and float(np.mean(confs)) < QWEN_TRIGGER_CONF:
        # Use the median-fused crop SR'd via bicubic ×4 as Qwen input
        aligned = ecc_register(select_sharp_frames(crops, K_FRAMES))
        fused = median_fuse(aligned)
        sr_for_qwen = sr_bicubic(fused)
        q_txt, q_conf = qwen_read_plate(sr_for_qwen, L, pat)
        if q_txt and len(q_txt) == L and q_conf > 0.6:
            qwen_used = True
            # Override only positions where ensemble conf < 0.55
            text = ''.join(q_txt[i] if confs[i] < 0.55 else text[i] for i in range(L))

    # NW alignment as a safety net (if length still wrong somehow)
    if len(text) != L:
        text = nw_align_to_template(text, pat)

    # Risk-aware abstention
    text = apply_abstention(text, confs, engine='ENSEMBLE', pat=pat)

    return text, {'L':L, 'pat':pat, 'mean_conf':float(np.mean(confs)), 'qwen':qwen_used,
                  'n_readings':len(readings_L)}

In [ ]:
# ============================================================
# 16 · Full dev-set evaluation
# ============================================================
rows = []
tot, mx_tot, exact = 0, 0, 0
t0 = time.time()
for i, seq in enumerate(SEQ_NAMES):
    gt = GT_DICT[seq].upper()
    pred, meta = predict_sequence(seq, gt_len_hint=len(gt))
    s, m = xlpsr_score(pred, gt)
    tot += s; mx_tot += m; exact += int(s == m)
    rows.append({'seq':seq, 'gt':gt, 'pred':pred, 'score':s, 'max':m, **meta})
    print(f'[{i+1:2d}/{len(SEQ_NAMES)}] {seq}  pred={pred:<8s}  gt={gt:<8s}  '
          f'{s:+3d}/{m}  qwen={meta.get("qwen",False)}  conf={meta.get("mean_conf",0):.2f}')

print(f'\nTOTAL: {tot:+d} / {mx_tot} = {100*tot/mx_tot:+.1f}%   exact={exact}/{len(SEQ_NAMES)}   '
      f'time={time.time()-t0:.0f}s')
df = pd.DataFrame(rows)

In [ ]:
# ============================================================
# 17 · Head-to-head vs predictions_ultraplate.csv (if present)
# ============================================================
prev_path = ROOT / 'predictions_ultraplate.csv'
if prev_path.exists():
    prev = pd.read_csv(prev_path).rename(columns={'sequence_id':'seq','predicted_lp':'prev'})
    cmp = df[['seq','gt','pred','score']].merge(prev, on='seq', how='left')
    cmp['prev_score'] = cmp.apply(lambda r: xlpsr_score(r['prev'], r['gt'])[0] if pd.notna(r['prev']) else 0, axis=1)
    cmp['delta'] = cmp['score'] - cmp['prev_score']
    print(f'ULTRAPLATE total : {cmp["prev_score"].sum():+d}')
    print(f'V2 total         : {cmp["score"].sum():+d}')
    print(f'Improvement      : {cmp["delta"].sum():+d}')
    print('\nTop wins:');   print(cmp.nlargest(5,'delta')[['seq','gt','prev','pred','delta']].to_string(index=False))
    print('\nRegressions:'); print(cmp.nsmallest(5,'delta')[['seq','gt','prev','pred','delta']].to_string(index=False))
else:
    print(f'No {prev_path.name} found — skipping head-to-head.')

In [ ]:
# ============================================================
# 18 · Submission CSV
# ============================================================
sub = df[['seq','pred']].rename(columns={'seq':'sequence_id','pred':'predicted_lp'})
sub.to_csv(OUT_CSV, index=False)
print(f'Wrote {OUT_CSV} ({len(sub)} rows)')
print(sub.to_string(index=False))

## 19 · Ablation knobs

Re-run cells 15-16 after toggling these flags in cell 1 to isolate each module's contribution:

| Knob | Off | On | Expected delta |
|---|---|---|---|
| `ENABLE_GRAMMAR_DECODE` | template enforced post-hoc only | template re-norms softmax during decode | +3 to +8 pts |
| `ENABLE_RISK_ABSTAIN` | static threshold (0.5) | isotonic per-engine calibration | +2 to +6 pts |
| `ENABLE_QWEN_VL` | ensemble alone | VLM adjudicator on shaky positions | +5 to +15 pts (largest single gain) |
| `ENABLE_BEST_OF_N` | argmax per-position | sampled top-50 ranked by joint score | +1 to +4 pts |
| `ENABLE_SR_*` flags | drop heaviest SR | use all 4 SR | +1 to +3 pts (diminishing) |

**Honest caveats:**
1. The risk calibration in cell 14 is fit on the dev set itself. For the CodaBench validation submission, set `CALIB_LOO=True` to use leave-one-out calibration, or fit on a held-out slice. Otherwise the dev numbers will overstate test performance.
2. The Qwen prompt was lightly tuned; a few failed sequences may benefit from a different phrasing (try removing the pattern hint and see).
3. Best-of-N with 50 candidates is fast on CPU but mostly redundant with the argmax for high-confidence plates. Lower to `BEST_OF_N=10` for a 5× speedup with nearly identical scores.

## 20 · Where this leaves us

If this notebook clears `ULTRAPLATE`'s −22 / 0-exact and lands above zero, the next zero-shot lever to pull is **multi-resolution OCR voting** (already mentioned in ultraplate §13E) — feed each engine the SR output at 48, 64, and 96 px heights and add those readings to the cross-product. Cheap to add (~10 lines), and orthogonal to everything here.

Beyond that, the fine-tuning plan in `/home/ibad/.claude/plans/hello-read-up-on-tender-pumpkin.md` is the next jump.